# Antelligence Torch Benchmark Worker (Colab)

Runs `compare_brains.py` + `check_torch_migration.py` on Colab and writes artifacts to Google Drive.

In [ ]:
# Parameters
REPO_URL = 'https://github.com/eren23/antelligence.git'
BRANCH = 'imp/learning-1'
MODE = 'torch_only'  # 'torch_only' or 'full'
TICKS = 5000
WARMUP_TICKS = 200
SEEDS = '42 123 7'
ANTS = 200
MAX_FOOD_DROP = 0.20
MAX_DEATH_INCREASE = 0.20
DRIVE_OUT_ROOT = '/content/drive/MyDrive/attocode_runs'
WORKDIR = '/content/work'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
from datetime import datetime

repo_dir = os.path.join(WORKDIR, 'repo')
os.makedirs(WORKDIR, exist_ok=True)

if not os.path.isdir(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, repo_dir], check=True)

subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=repo_dir, check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=repo_dir, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=repo_dir, check=True)

ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out_dir = os.path.join(DRIVE_OUT_ROOT, f'bench_{ts}')
os.makedirs(out_dir, exist_ok=True)

cmd = [
    'bash', 'scripts/colab_benchmark.sh',
    '--mode', MODE,
    '--ticks', str(TICKS),
    '--warmup-ticks', str(WARMUP_TICKS),
    '--seeds', SEEDS,
    '--ants', str(ANTS),
    '--max-food-drop', str(MAX_FOOD_DROP),
    '--max-death-increase', str(MAX_DEATH_INCREASE),
    '--out-dir', out_dir,
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)
print('Done. Artifacts in:', out_dir)


In [ ]:
# Optional: view the migration check summary
import json, os
summary_path = os.path.join(out_dir, 'migration_check.json')
with open(summary_path) as f:
    print(json.dumps(json.load(f), indent=2))
